In [ ]:
from openadmet.toolkit.database.chembl import PermissiveChEMBLTargetCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm
import os
import subprocess

In [ ]:
def gather_chembl_data_for_target(target_name: str, chembl_tid: str, chembl_ver: int):
    print(f"working on target {target_name}")
    pctc = PermissiveChEMBLTargetCurator(chembl_target_id=chembl_tid, version=chembl_ver, standard_type="EC50", require_pchembl=True)
    activity_data = pctc.get_activity_data(return_as="df")

    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data

In [ ]:
targets = {
    "AHR": "CHEMBL3201",
    "PXR": "CHEMBL3401",
}

In [ ]:
chembl_ver = 35

In [ ]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_access_key_id"],
    text=True
).strip()

os.environ["AWS_SECRET_ACCESS_KEY"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_secret_access_key"],
    text=True
).strip()

In [ ]:
settings = S3Settings()

In [ ]:
bucket = "openadmet-data-public-dev"

In [ ]:
bucket = S3Bucket.from_settings(settings, bucket)

In [ ]:
import datetime

In [ ]:
t = datetime.datetime.now()

In [ ]:
date = t.strftime("%Y-%m-%d")

In [ ]:
location=f"ChEMBL{chembl_ver}_EC50"

In [ ]:
import os
from pathlib import Path

location_path = Path(location)

In [ ]:
location_path.mkdir(exist_ok=False)

In [ ]:
uris_raw = {}
uris_agg = {}
for target, chembl_tid in targets.items():

    agg, raw  = gather_chembl_data_for_target(target, chembl_tid, chembl_ver)
    # TODO: make a function this is clunky
    fname_agg = f"ChEMBL_EC50_{target}_{chembl_tid}_aggregated.parquet"
    fname_raw = f"ChEMBL_EC50_{target}_{chembl_tid}_raw.parquet"
    
    agg.reset_index(drop=True).to_parquet(location_path/fname_agg, index=False)
    raw.reset_index(drop=True).to_parquet(location_path/fname_raw, index=False)
    
    bucket_destination_agg = location + "/" + fname_agg
    bucket.push_file(location_path/fname_agg, bucket_destination_agg)
    bucket_destination_raw = location + "/" + fname_raw
    bucket.push_file(location_path/fname_raw, bucket_destination_raw)

    # get S3 URIs
    uri_agg = bucket.to_uri(bucket_destination_agg)
    uris_agg[target] = uri_agg

    uri_raw = bucket.to_uri(bucket_destination_raw)
    uris_raw[target] = uri_raw

In [ ]:
import intake
intake.Catalog?
cat = intake.entry.Catalog()

In [ ]:
uris_agg

In [ ]:
uris_raw

In [ ]:
for k,v in uris_agg.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

In [ ]:
for k,v in uris_raw.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

In [ ]:
catname = f"CATALOG_{location}.yaml"

In [ ]:
cat.to_yaml_file(catname)

In [ ]:
cat_location = location+ "/" +catname

In [ ]:
cat_location

In [ ]:
bucket.push_file(catname, cat_location)

In [ ]:
cat_uri = bucket.to_uri(cat_location)